## EDA for text

Cell cluster distribution and class imbalance<br>
UMAP visualization<br>
Spatial coordinate distribution<br>
Transcript quantity (data quality)<br>
Marker gene heatmap<br>

In [1]:
# Breast Cancer Xenium Data - EDA
# Run on Kaggle: attach your dataset and update BASE_PATH below

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
import os
import warnings
warnings.filterwarnings('ignore')

In [2]:
# SECTION 0: CONFIG
import os

EXCEL_PATH           = '/kaggle/input/datasets/eva233333/data3888-grp/41467_2023_43458_MOESM4_ESM.xlsx'
CBR_PATH             = '/kaggle/input/datasets/eva233333/data3888-grp/cbr.csv'
CELL_BOUNDARIES_PATH = '/kaggle/input/datasets/eva233333/data3888-grp/cell_boundaries.csv'
THREEPTS_PATH        = '/kaggle/input/datasets/eva233333/data3888-grp/threepts.csv'
HISTOLOGY_PATH       = '/kaggle/input/datasets/eva233333/data3888-grp/GSM7780153_Post-Xenium_HE_Rep1.ome.tif'

# 100px image path
IMAGE_DIRS_100 = {
    'DCIS_1':                '/kaggle/input/datasets/eva233333/tumor-cell-dcis-2/DCIS_1/DCIS_1/',
    'DCIS_2':                '/kaggle/input/datasets/eva233333/tumor-cell-dcis-2/DCIS_2/DCIS_2/',
    'Prolif_Invasive_Tumor': '/kaggle/input/datasets/eva233333/tumor-cell-dcis-2/Prolif_Invasive_Tumor/Prolif_Invasive_Tumor/',
    'Invasive_Tumor':        '/kaggle/input/datasets/eva233333/tumor-cell-dcis-2/Invasive_Tumor/Invasive_Tumor/',
}

# 50px image path
IMAGE_DIRS_50 = {
    'DCIS_1':                '/kaggle/input/datasets/eva233333/50px-tumor-cell/DCIS_1/DCIS_1/',
    'DCIS_2':                '/kaggle/input/datasets/eva233333/50px-tumor-cell/DCIS_2/DCIS_2/',
    'Prolif_Invasive_Tumor': '/kaggle/input/datasets/eva233333/50px-tumor-cell/Prolif_Invasive_Tumor (1)/Prolif_Invasive_Tumor/',
    'Invasive_Tumor':        '/kaggle/input/datasets/eva233333/50px-tumor-cell/Invasive_Tumor (1)/Invasive_Tumor/',
}

LABEL_MAP = {
    'DCIS_1': 0,
    'DCIS_2': 1,
    'Prolif_Invasive_Tumor': 2,
    'Invasive_Tumor': 3,
}

TUMOR_TYPES_XENIUM = ['DCIS 1', 'DCIS 2', 'Prolif_Invasive_Tumor', 'Invasive_Tumor']

COLORS = {
    'DCIS_1':                '#E63946',
    'DCIS_2':                '#F4A261',
    'Prolif_Invasive_Tumor': '#2A9D8F',
    'Invasive_Tumor':        '#457B9D',
}

OUT_DIR = '/kaggle/working/'

def get_cell_id(filename):
    """'cell_158510_100.png' -> 158510"""
    return int(filename.split('_')[1])


print("\n 100px File number verification")
for cl, path in IMAGE_DIRS_100.items():
    n = len(os.listdir(path)) if os.path.exists(path) else "PATH NOT FOUND"
    print(f"  {cl:30s}: {n}")

print("\n 50px File number verification")
for cl, path in IMAGE_DIRS_50.items():
    n = len(os.listdir(path)) if os.path.exists(path) else "PATH NOT FOUND"
    print(f"  {cl:30s}: {n}")


 100px File number verification
  DCIS_1                        : 12925
  DCIS_2                        : 11719
  Prolif_Invasive_Tumor         : 3775
  Invasive_Tumor                : 34398

 50px File number verification
  DCIS_1                        : 12923
  DCIS_2                        : 11683
  Prolif_Invasive_Tumor         : 3775
  Invasive_Tumor                : 34374


In [3]:
# SECTION 1: LOAD DATA

xl = pd.read_excel(EXCEL_PATH, sheet_name=None)
print(f"Sheets found: {list(xl.keys())}")

xenium   = xl['Fig. 3e-j Xenium']        # main single-cell data
scffpe   = xl['Fig. 2a scFFPE-seq UMAP'] # scRNA-seq reference
heatmap  = xl['Fig. 3k Heatmap']         # marker gene expression

print(f"\nXenium shape:  {xenium.shape}")
print(f"scFFPE shape:  {scffpe.shape}")
print(f"Heatmap shape: {heatmap.shape}")
print("\nXenium columns:", list(xenium.columns))
print("\nXenium dtypes:\n", xenium.dtypes)


Sheets found: ['Fig. 2a scFFPE-seq UMAP', 'Fig. 3e-j Xenium', 'Fig. 3k Heatmap', 'Fig. 4a ROIs', 'Fig. 4b-d ', 'Fig. 5 Spot Binned Xenium Data', 'Fig. 6e Heatmap', 'Fig. 6f Violins', 'Sup. Fig. 1 Panel Heatmap', 'Sup. Fig. 2', 'Sup. Fig. 3 Flex Heatmap', 'Sup. Fig. 5 Scatter', 'Sup Fig. 5 Xenium Rep 2', 'Sup. Fig. 7 SC Comparison', 'Sup. Fig. 10 Visium Deconv.']

Xenium shape:  (167780, 8)
scFFPE shape:  (27472, 4)
Heatmap shape: (313, 21)

Xenium columns: ['Barcode', 'UMAP_DIM1', 'UMAP_DIM2', 'Cluster', 'transcript_counts', 'x_centroid', 'y_centroid', 'gene_counts']

Xenium dtypes:
 Barcode                int64
UMAP_DIM1            float64
UMAP_DIM2            float64
Cluster               object
transcript_counts      int64
x_centroid           float64
y_centroid           float64
gene_counts            int64
dtype: object


In [4]:
# SECTION 2: BASIC STATS

print("\n All cluster counts")
print(xenium['Cluster'].value_counts().to_string())

print("\n Missing values in Xenium")
print(xenium.isnull().sum())

print("\n Coordinate ranges")
print(f"x_centroid: {xenium['x_centroid'].min():.1f} → {xenium['x_centroid'].max():.1f}")
print(f"y_centroid: {xenium['y_centroid'].min():.1f} → {xenium['y_centroid'].max():.1f}")

print("\n Quality metrics (all cells)")
print(xenium[['transcript_counts', 'gene_counts']].describe().round(1))

# Cells with 0 transcripts
zero_tx = xenium[xenium['transcript_counts'] == 0]
print(f"\nCells with 0 transcripts: {len(zero_tx)}")
print(zero_tx['Cluster'].value_counts())


 All cluster counts
Cluster
Stromal                    41422
Invasive_Tumor             34374
DCIS 1                     12923
DCIS 2                     11683
Macrophages_1              11174
Endothelial                 8931
Unlabeled                   8554
CD4+_T_Cells                8453
Myoepi_ACTA2+               7078
CD8+_T_Cells                6940
B_Cells                     4987
Prolif_Invasive_Tumor       3775
Myoepi_KRT15+               2860
Macrophages_2               1624
Perivascular-Like            847
Stromal_&_T_Cell_Hybrid      607
T_Cell_&_Tumor_Hybrid        589
IRF7+_DCs                    494
LAMP3+_DCs                   298
Mast_Cells                   167

 Missing values in Xenium
Barcode                 0
UMAP_DIM1            7478
UMAP_DIM2            7478
Cluster                 0
transcript_counts       0
x_centroid              0
y_centroid              0
gene_counts             0
dtype: int64

 Coordinate ranges
x_centroid: 2.1 → 7523.1
y_centroid: 1.4 → 

In [5]:
# SECTION 3: TUMOR CELL ANALYSIS
TUMOR_TYPES = ['DCIS 1', 'DCIS 2', 'Prolif_Invasive_Tumor', 'Invasive_Tumor']  # Xenium表里用空格

tumor_df = xenium[xenium['Cluster'].isin(TUMOR_TYPES)].copy()
counts   = tumor_df['Cluster'].value_counts()

print(f"\n Tumor cell counts")
print(counts)
print(f"\nTotal tumor cells: {len(tumor_df)}")
print(f"Class imbalance (max/min): {counts.max()/counts.min():.1f}x")

print("\n Transcript counts per tumor cluster")
print(tumor_df.groupby('Cluster')['transcript_counts'].describe().round(1))

print("\n Spatial extent per cluster")
for cl in TUMOR_TYPES:
    sub = tumor_df[tumor_df['Cluster'] == cl]
    print(f"{cl:30s}: n={len(sub):5d} | "
          f"x=[{sub.x_centroid.min():.0f},{sub.x_centroid.max():.0f}] "
          f"y=[{sub.y_centroid.min():.0f},{sub.y_centroid.max():.0f}]")

# Naming inconsistency warning
print("\n Cluster name formats (watch for space vs underscore!)")
for c in sorted(xenium['Cluster'].unique()):
    print(repr(c))




 Tumor cell counts
Cluster
Invasive_Tumor           34374
DCIS 1                   12923
DCIS 2                   11683
Prolif_Invasive_Tumor     3775
Name: count, dtype: int64

Total tumor cells: 62755
Class imbalance (max/min): 9.1x

 Transcript counts per tumor cluster
                         count   mean    std   min    25%    50%    75%  \
Cluster                                                                   
DCIS 1                 12923.0  284.0  132.4  11.0  189.0  271.0  364.0   
DCIS 2                 11683.0  216.7  128.8  11.0  125.0  200.0  285.0   
Invasive_Tumor         34374.0  232.0  129.7  11.0  138.0  214.0  305.0   
Prolif_Invasive_Tumor   3775.0  307.4  150.7  20.0  195.0  288.0  394.0   

                          max  
Cluster                        
DCIS 1                 1329.0  
DCIS 2                 1259.0  
Invasive_Tumor         1181.0  
Prolif_Invasive_Tumor  1029.0  

 Spatial extent per cluster
DCIS 1                        : n=12923 | x=[3,7521] y

In [6]:
# SECTION 4: IMAGE FILE EXPLORATION

for size_label, image_dirs in [('50px', IMAGE_DIRS_50), ('100px', IMAGE_DIRS_100)]:
    print(f"\n {size_label}")
    for cl, path in image_dirs.items():
        p = Path(path)
        if not p.exists():
            print(f"  WARNING: {path} not found")
            continue
        imgs = list(p.glob('*.png'))
        print(f"  {cl:30s}: {len(imgs):6d} images")
        if imgs:
            for img in imgs[:2]:
                print(f"    example: {img.name}")


 50px
  DCIS_1                        :  12923 images
    example: cell_138442_50.png
    example: cell_153762_50.png
  DCIS_2                        :  11683 images
    example: cell_38236_50.png
    example: cell_67373_50.png
  Prolif_Invasive_Tumor         :   3775 images
    example: cell_71291_50.png
    example: cell_64497_50.png
  Invasive_Tumor                :  34374 images
    example: cell_22488_50.png
    example: cell_73465_50.png

 100px
  DCIS_1                        :  12925 images
    example: cell_130874_100.png
    example: cell_161072_100.png
  DCIS_2                        :  11719 images
    example: cell_158510_100.png
    example: cell_46750_100.png
  Prolif_Invasive_Tumor         :   3775 images
    example: cell_59836_100.png
    example: cell_17266_100.png
  Invasive_Tumor                :  34398 images
    example: cell_24387_100.png
    example: cell_72695_100.png


In [7]:
# SECTION 5: BARCODE <-> FILENAME MATCHING

# Try to figure out if image filenames contain barcodes
import pandas as pd

xenium = pd.read_excel(EXCEL_PATH, sheet_name='Fig. 3e-j Xenium')
xenium['Cluster'] = xenium['Cluster'].str.replace(' ', '_')
xenium['cell_id'] = range(1, len(xenium) + 1)  # cell_id是1-based行号

print(f"Total number of cells in Xenium: {len(xenium)}")
print(f"cell_id range: {xenium['cell_id'].min()} - {xenium['cell_id'].max()}")

# Verify using the sample file of DCIS_1
sample_dir = IMAGE_DIRS_100['DCIS_1']
sample_files = os.listdir(sample_dir)[:10]

print("\n Verification result (cell_id -> Xenium coordinates):")
all_matched = True
for fname in sample_files:
    cid = get_cell_id(fname)
    row = xenium[xenium['cell_id'] == cid]
    if len(row) > 0:
        r = row.iloc[0]
        print(f"  {fname:30s} -> cluster={r['Cluster']:25s} x={r['x_centroid']:.1f} y={r['y_centroid']:.1f} ✓")
    else:
        print(f"  {fname:30s} -> NOT FOUND ✗")
        all_matched = False

Total number of cells in Xenium: 167780
cell_id range: 1 - 167780

 Verification result (cell_id -> Xenium coordinates):
  cell_130874_100.png            -> cluster=DCIS_1                    x=5119.4 y=3357.5 ✓
  cell_161072_100.png            -> cluster=DCIS_1                    x=7062.4 y=3457.1 ✓
  cell_133340_100.png            -> cluster=DCIS_1                    x=5385.7 y=3460.9 ✓
  cell_154812_100.png            -> cluster=DCIS_1                    x=7052.5 y=2132.1 ✓
  cell_167130_100.png            -> cluster=DCIS_1                    x=7452.2 y=3314.7 ✓
  cell_157898_100.png            -> cluster=DCIS_1                    x=7124.3 y=3022.4 ✓
  cell_153864_100.png            -> cluster=DCIS_1                    x=7246.3 y=2190.9 ✓
  cell_165906_100.png            -> cluster=DCIS_1                    x=7399.8 y=5102.9 ✓
  cell_13941_100.png             -> cluster=DCIS_1                    x=7510.7 y=3356.6 ✓
  cell_162275_100.png            -> cluster=DCIS_1                   

In [8]:
# SECTION 6: VISUALISATIONS
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Unified Cluster Naming (Spaces → Underscores)
xenium['Cluster'] = xenium['Cluster'].str.replace(' ', '_')

TUMOR_TYPES = ['DCIS_1', 'DCIS_2', 'Prolif_Invasive_Tumor', 'Invasive_Tumor']
COLORS = {
    'DCIS_1':                '#E63946',
    'DCIS_2':                '#F4A261',
    'Prolif_Invasive_Tumor': '#2A9D8F',
    'Invasive_Tumor':        '#457B9D',
}
tumor_df = xenium[xenium['Cluster'].isin(TUMOR_TYPES)].copy()
counts   = tumor_df['Cluster'].value_counts()
patches  = [mpatches.Patch(color=COLORS[c], label=c) for c in TUMOR_TYPES]

# Figure 1: Main EDA overview (6 panels) 
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('EDA: Xenium Breast Cancer Single-Cell Data', fontsize=16, fontweight='bold')

# Panel 1: All cluster sizes
ax = axes[0, 0]
all_counts  = xenium['Cluster'].value_counts()
bar_colors  = ['#E63946' if c in TUMOR_TYPES else '#AAAAAA' for c in all_counts.index]
ax.barh(range(len(all_counts)), all_counts.values, color=bar_colors)
ax.set_yticks(range(len(all_counts)))
ax.set_yticklabels(all_counts.index, fontsize=8)
ax.set_xlabel('Cell Count')
ax.set_title('All Cluster Sizes\n(red = selected tumor types)', fontweight='bold')
ax.invert_yaxis()

# Panel 2: Tumor class imbalance
ax = axes[0, 1]
bar_cols = [COLORS[c] for c in counts.index]
ax.bar(range(len(counts)), counts.values, color=bar_cols, edgecolor='white', width=0.6)
ax.set_xticks(range(len(counts)))
ax.set_xticklabels([c.replace('_', '\n') for c in counts.index], fontsize=9)
ax.set_ylabel('Cell Count')
ax.set_title('Tumor Cell Class Imbalance\n(9.1× max/min ratio)', fontweight='bold')
for i, v in enumerate(counts.values):
    ax.text(i, v + 200, f'{v:,}', ha='center', fontsize=9, fontweight='bold')

# Panel 3: UMAP — all cells, tumor highlighted
ax = axes[0, 2]
non_tumor = xenium[~xenium['Cluster'].isin(TUMOR_TYPES)].dropna(subset=['UMAP_DIM1'])
ax.scatter(non_tumor['UMAP_DIM1'], non_tumor['UMAP_DIM2'],
           c='#DDDDDD', s=0.3, alpha=0.3, rasterized=True)
for cl in TUMOR_TYPES:
    sub = xenium[xenium['Cluster'] == cl].dropna(subset=['UMAP_DIM1'])
    ax.scatter(sub['UMAP_DIM1'], sub['UMAP_DIM2'],
               c=COLORS[cl], s=0.5, alpha=0.6, label=cl, rasterized=True)
ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
ax.set_title('UMAP (Xenium)\nTumor clusters highlighted', fontweight='bold')
patches = [mpatches.Patch(color=COLORS[c], label=c) for c in TUMOR_TYPES]
ax.legend(handles=patches, fontsize=7, loc='lower right')

# Panel 4: Spatial map
ax = axes[1, 0]
non_t = xenium[~xenium['Cluster'].isin(TUMOR_TYPES)]
ax.scatter(non_t['x_centroid'], non_t['y_centroid'],
           c='#EEEEEE', s=0.05, alpha=0.2, rasterized=True)
for cl in TUMOR_TYPES:
    sub = xenium[xenium['Cluster'] == cl]
    ax.scatter(sub['x_centroid'], sub['y_centroid'],
               c=COLORS[cl], s=0.2, alpha=0.5, rasterized=True)
ax.set_xlabel('X centroid (µm)'); ax.set_ylabel('Y centroid (µm)')
ax.set_title('Spatial Map of Tumour Cells\n(grey = other cell types)', fontweight='bold')
ax.legend(handles=patches, fontsize=7)
ax.invert_yaxis()

# Panel 5: Transcript count distribution
ax = axes[1, 1]
for cl in TUMOR_TYPES:
    sub = tumor_df[tumor_df['Cluster'] == cl]['transcript_counts']
    ax.hist(sub, bins=60, alpha=0.5, color=COLORS[cl], label=cl, density=True)
ax.set_xlabel('Transcript Count'); ax.set_ylabel('Density')
ax.set_title('Transcript Count per Cluster\n(proxy for data quality)', fontweight='bold')
ax.legend(fontsize=7)
ax.set_xlim(0, 900)

# Panel 6: Marker gene heatmap
ax = axes[1, 2]
tumor_cols_hm = ['DCIS_1', 'DCIS_2', 'Prolif_Invasive_Tumor', 'Invasive_Tumor']
hm = heatmap[['Genes'] + tumor_cols_hm].set_index('Genes')
hm['_max'] = hm.abs().max(axis=1)
top15 = hm.nlargest(15, '_max').drop('_max', axis=1)
im = ax.imshow(top15.values.T, aspect='auto', cmap='RdBu_r', vmin=-2, vmax=2)
ax.set_xticks(range(len(top15.index)))
ax.set_xticklabels(top15.index, rotation=45, ha='right', fontsize=7)
ax.set_yticks(range(len(tumor_cols_hm)))
ax.set_yticklabels([c.replace('_', ' ') for c in tumor_cols_hm], fontsize=8)
ax.set_title('Top 15 Marker Genes\n(tumor clusters)', fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.7, label='Scaled Expression')

plt.tight_layout()
out1 = OUT_DIR + 'fig1_eda_overview.png'
plt.savefig(out1, dpi=150, bbox_inches='tight')
plt.close()


# Figure 2: UMAP side-by-side comparison (scFFPE vs Xenium) 
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('UMAP Comparison: scFFPE-seq vs Xenium', fontsize=14, fontweight='bold')

for ax, df, dim1, dim2, label_col, title in [
    (axes[0], scffpe,  'UMAP-X',    'UMAP-Y',    'Annotation', 'scFFPE-seq'),
    (axes[1], xenium,  'UMAP_DIM1', 'UMAP_DIM2', 'Cluster',    'Xenium'),
]:
    df_valid = df.dropna(subset=[dim1, dim2])
    non_t = df_valid[~df_valid[label_col].isin(TUMOR_TYPES)]
    ax.scatter(non_t[dim1], non_t[dim2], c='#DDDDDD', s=0.3, alpha=0.3, rasterized=True)
    for cl in TUMOR_TYPES:
        sub = df_valid[df_valid[label_col] == cl]
        if len(sub) > 0:
            ax.scatter(sub[dim1], sub[dim2], c=COLORS[cl], s=0.5, alpha=0.7,
                       label=cl, rasterized=True)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel(dim1); ax.set_ylabel(dim2)

axes[1].legend(handles=patches, fontsize=8, loc='lower right')
plt.tight_layout()
out2 = OUT_DIR + 'fig2_umap_comparison.png'
plt.savefig(out2, dpi=150, bbox_inches='tight')
plt.close()


# Figure 3: Spatial map — all 20 cluster types 
fig, ax = plt.subplots(figsize=(12, 9))
unique_clusters = xenium['Cluster'].unique()
cmap = plt.cm.get_cmap('tab20', len(unique_clusters))
cluster_color_map = {cl: cmap(i) for i, cl in enumerate(unique_clusters)}

for cl in unique_clusters:
    sub = xenium[xenium['Cluster'] == cl]
    size = 0.5 if cl in TUMOR_TYPES else 0.1
    alpha = 0.7 if cl in TUMOR_TYPES else 0.2
    ax.scatter(sub['x_centroid'], sub['y_centroid'],
               c=[cluster_color_map[cl]], s=size, alpha=alpha,
               label=cl, rasterized=True)

ax.set_xlabel('X centroid (µm)'); ax.set_ylabel('Y centroid (µm)')
ax.set_title('Full Tissue Spatial Map — All Cell Types', fontweight='bold')
ax.invert_yaxis()
ax.legend(markerscale=8, fontsize=6, loc='upper right',
          ncol=2, bbox_to_anchor=(1.25, 1))
plt.tight_layout()
out3 = OUT_DIR + 'fig3_spatial_all_clusters.png'
plt.savefig(out3, dpi=150, bbox_inches='tight', bbox_extra_artists=[ax.legend()])
plt.close()


# Figure 4: Nearest-neighbour composition (microenvironment preview)
from sklearn.neighbors import BallTree

coords = xenium[['x_centroid', 'y_centroid']].values
tree   = BallTree(np.radians(coords) if False else coords)  # Euclidean

# For each tumor cell, find k=10 neighbours and record their cluster composition
K = 10
tumor_idx = xenium[xenium['Cluster'].isin(TUMOR_TYPES)].index.tolist()
# Use a subsample for speed (5000 cells)
rng = np.random.default_rng(42)
sample_idx = rng.choice(tumor_idx, size=min(5000, len(tumor_idx)), replace=False)

all_clusters = xenium['Cluster'].values
neighbour_compositions = {cl: {t: 0 for t in TUMOR_TYPES} for cl in TUMOR_TYPES}

sample_pos = xenium.loc[sample_idx, ['x_centroid', 'y_centroid']].values
_, nn_indices = tree.query(sample_pos, k=K+1)  # +1 because cell itself is included

for i, idx in enumerate(sample_idx):
    focal_cluster = xenium.loc[idx, 'Cluster']
    neighbours    = nn_indices[i, 1:]  # exclude self
    neighbour_clusters = all_clusters[neighbours]
    for nc in neighbour_clusters:
        if nc in TUMOR_TYPES:
            neighbour_compositions[focal_cluster][nc] += 1

# Normalise to proportions
fig, ax = plt.subplots(figsize=(8, 5))
comp_df = pd.DataFrame(neighbour_compositions).T
comp_df = comp_df.div(comp_df.sum(axis=1), axis=0)
comp_df.plot(kind='bar', ax=ax, color=[COLORS[c] for c in comp_df.columns],
             edgecolor='white', width=0.7)
ax.set_xlabel('Focal Cell Cluster')
ax.set_ylabel('Proportion of Tumor Neighbours')
ax.set_title(f'Tumour Microenvironment Composition\n(k={K} nearest neighbours, n=5000 sample)',
             fontweight='bold')
ax.legend(title='Neighbour type', fontsize=8)
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right')
plt.tight_layout()
out4 = OUT_DIR + 'fig4_microenvironment.png'
plt.savefig(out4, dpi=150, bbox_inches='tight')
plt.close()




In [9]:
# SECTION 7: EXPORT CLEAN LABEL TABLE

clean = tumor_df[['Barcode', 'Cluster', 'x_centroid', 'y_centroid',
                   'transcript_counts', 'gene_counts']].copy()

# Standardise cluster names: replace space with underscore
clean['Cluster'] = clean['Cluster'].str.replace(' ', '_')

# Add numeric label for ML
label_map = {
    'DCIS_1': 0,
    'DCIS_2': 1,
    'Prolif_Invasive_Tumor': 2,
    'Invasive_Tumor': 3,
}
clean['label'] = clean['Cluster'].map(label_map)

# Basic quality filter: keep cells with at least 10 transcripts
before = len(clean)
clean  = clean[clean['transcript_counts'] >= 10]
print(f"  Filtered {before - len(clean)} low-quality cells (transcript_count < 10)")
print(f"  Final table: {len(clean)} cells")
print(clean.head())

out_csv = OUT_DIR + 'tumor_cells_clean.csv'
clean.to_csv(out_csv, index=False)


  Filtered 0 low-quality cells (transcript_count < 10)
  Final table: 62755 cells
   Barcode         Cluster  x_centroid  y_centroid  transcript_counts  \
0        1          DCIS_2  847.259912  326.191365                 28   
1        2          DCIS_2  826.341995  328.031830                 94   
3        4  Invasive_Tumor  824.228409  334.252643                 11   
4        5          DCIS_2  841.357538  332.242505                 48   
7        8          DCIS_2  828.726239  341.712347                 39   

   gene_counts  label  
0           15      1  
1           38      1  
3            9      3  
4           33      1  
7           25      1  


## EDA for images

In [10]:
# IMAGE EDA + PREPROCESSING
import os, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from pathlib import Path
from collections import defaultdict
warnings.filterwarnings('ignore')

### PART 1 — BASIC EDA

In [11]:
# IMAGE_DIRS_100, IMAGE_DIRS_50, LABEL_MAP, COLORS, OUT_DIR, get_cell_id
SEED = 3888
random.seed(SEED)
np.random.seed(SEED)

#### 1-A  Sample counts & class distribution

In [12]:
records_100, records_50 = [], []

for cl, path in IMAGE_DIRS_100.items():
    files = [f for f in os.listdir(path) if f.endswith('.png')]
    for f in files:
        records_100.append({'cluster': cl, 'label': LABEL_MAP[cl],
                            'cell_id': get_cell_id(f), 'path': os.path.join(path, f)})

for cl, path in IMAGE_DIRS_50.items():
    files = [f for f in os.listdir(path) if f.endswith('.png')]
    for f in files:
        records_50.append({'cluster': cl, 'label': LABEL_MAP[cl],
                           'cell_id': get_cell_id(f), 'path': os.path.join(path, f)})

df100 = pd.DataFrame(records_100)
df50  = pd.DataFrame(records_50)

# Intersection (cells with both sizes)
ids_both = {}
for cl in IMAGE_DIRS_100.keys():
    ids_100 = set(df100[df100['cluster'] == cl]['cell_id'])
    ids_50  = set(df50[df50['cluster']  == cl]['cell_id'])
    ids_both[cl] = ids_100 & ids_50

df_both = pd.concat([
    df100[(df100['cluster'] == cl) & (df100['cell_id'].isin(ids))]
    for cl, ids in ids_both.items()
]).reset_index(drop=True)

counts_100  = df100['cluster'].value_counts().sort_index()
counts_50   = df50['cluster'].value_counts().sort_index()
counts_both = df_both['cluster'].value_counts().sort_index()

print(f"\n{'Cluster':<30} {'100px':>8} {'50px':>8} {'Both':>8}")
print("-" * 58)
for cl in sorted(IMAGE_DIRS_100.keys()):
    print(f"{cl:<30} {counts_100.get(cl,0):>8,} {counts_50.get(cl,0):>8,} {counts_both.get(cl,0):>8,}")
print(f"{'TOTAL':<30} {len(df100):>8,} {len(df50):>8,} {len(df_both):>8,}")

imbalance = counts_both.max() / counts_both.min()
print(f"\nClass imbalance ratio (max/min): {imbalance:.1f}x")

# Plot
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('1-A: Sample Counts & Class Distribution', fontweight='bold')

for ax, counts, title in zip(axes,
    [counts_100, counts_50, counts_both],
    ['100px', '50px', 'Intersection (both)']):
    bars = ax.bar(range(len(counts)), counts.values,
                  color=[COLORS[c] for c in counts.index], edgecolor='white')
    ax.set_xticks(range(len(counts)))
    ax.set_xticklabels([c.replace('_', '\n') for c in counts.index], fontsize=8)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Cell Count')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 100, f'{v:,}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig(OUT_DIR + 'basic_1a_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()



Cluster                           100px     50px     Both
----------------------------------------------------------
DCIS_1                           12,925   12,923   12,925
DCIS_2                           11,719   11,683   11,719
Invasive_Tumor                   34,398   34,374   34,398
Prolif_Invasive_Tumor             3,775    3,775    3,775
TOTAL                            62,817   62,755   62,817

Class imbalance ratio (max/min): 9.1x


#### # 1-B  Image size distribution (width, height, aspect ratio)

In [13]:
size_records = []
SAMPLE_N = 200  # sample per cluster for speed

for cl, path in IMAGE_DIRS_100.items():
    files = random.sample(os.listdir(path), min(SAMPLE_N, len(os.listdir(path))))
    for fname in files:
        img = Image.open(os.path.join(path, fname))
        w, h = img.size
        size_records.append({'cluster': cl, 'size': '100px', 'width': w, 'height': h,
                              'aspect': w / h})

for cl, path in IMAGE_DIRS_50.items():
    files = random.sample(os.listdir(path), min(SAMPLE_N, len(os.listdir(path))))
    for fname in files:
        img = Image.open(os.path.join(path, fname))
        w, h = img.size
        size_records.append({'cluster': cl, 'size': '50px', 'width': w, 'height': h,
                              'aspect': w / h})

size_df = pd.DataFrame(size_records)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('1-B: Image Size Distribution', fontweight='bold')

for ax, col, xlabel in zip(axes,
    ['width', 'height', 'aspect'],
    ['Width (px)', 'Height (px)', 'Aspect Ratio (W/H)']):
    for sz, color in [('50px', '#457B9D'), ('100px', '#E63946')]:
        vals = size_df[size_df['size'] == sz][col]
        ax.hist(vals, bins=30, alpha=0.6, label=sz, color=color, density=True)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Density')
    ax.set_title(col, fontweight='bold')
    ax.legend()

plt.tight_layout()
plt.savefig(OUT_DIR + 'basic_1b_image_sizes.png', dpi=150, bbox_inches='tight')
plt.show()


#### 1-C RGB channel distribution

In [14]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os, random

def get_rgb_stats(path, sample_n=200):
    files = random.sample(os.listdir(path), min(sample_n, len(os.listdir(path))))
    r_means, g_means, b_means = [], [], []
    for fname in files:
        img = np.array(Image.open(os.path.join(path, fname)).convert('RGB'))
        r_means.append(img[:,:,0].mean())
        g_means.append(img[:,:,1].mean())
        b_means.append(img[:,:,2].mean())
    return np.array(r_means), np.array(g_means), np.array(b_means)


def mean_image(path, sample_n=500):
    files = random.sample(os.listdir(path), min(sample_n, len(os.listdir(path))))
    imgs = [np.array(Image.open(os.path.join(path, f)).convert('RGB').resize((100,100))) 
            for f in files]
    return np.mean(imgs, axis=0).astype(np.uint8)

random.seed(42)

# collect RGB data
rgb_data = {}
for cl, path in IMAGE_DIRS_100.items():
    r, g, b = get_rgb_stats(path)
    rgb_data[cl] = {'R': r, 'G': g, 'B': b}

# Violin plot
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('RGB Channel Distribution by Cluster (100px)', fontweight='bold')

for ax, ch in zip(axes, ['R', 'G', 'B']):
    data = [rgb_data[cl][ch] for cl in IMAGE_DIRS_100.keys()]
    parts = ax.violinplot(data, positions=range(4), showmedians=True)
    for i, pc in enumerate(parts['bodies']):
        pc.set_facecolor(list(COLORS.values())[i])
        pc.set_alpha(0.7)
    ax.set_xticks(range(4))
    ax.set_xticklabels([c[:6] for c in IMAGE_DIRS_100.keys()], rotation=30, fontsize=8)
    ax.set_title(f'{ch} channel', fontweight='bold')
    ax.set_ylabel('Mean pixel value')

plt.tight_layout()
plt.savefig(OUT_DIR + 'rgb_violin.png', dpi=150, bbox_inches='tight')
plt.show()

# Mean image grid
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('Mean Image per Cluster (100px)', fontweight='bold')

for ax, (cl, path) in zip(axes, IMAGE_DIRS_100.items()):
    mi = mean_image(path)
    ax.imshow(mi)
    ax.set_title(cl.replace('_', '\n'), fontweight='bold', color=COLORS[cl])
    ax.axis('off')

plt.tight_layout()
plt.savefig(OUT_DIR + 'rgb_mean_image.png', dpi=150, bbox_inches='tight')
plt.show()

### PART 2 — DEEPER IMAGE EDA


#### 2-A  Tissue content (non-white pixel ratio)

In [15]:
# 2-A  Tissue content (non-white pixel ratio)
def tissue_ratio(img_path):
    img = np.array(Image.open(img_path).convert('RGB'))
    return 1 - np.all(img > 230, axis=-1).mean()

tissue_records = []

for cl, path in IMAGE_DIRS_100.items():
    files = random.sample(os.listdir(path), min(SAMPLE_N, len(os.listdir(path))))
    for fname in files:
        ratio = tissue_ratio(os.path.join(path, fname))
        tissue_records.append({'cluster': cl, 'size': '100px', 'tissue_ratio': ratio})

for cl, path in IMAGE_DIRS_50.items():
    files = random.sample(os.listdir(path), min(SAMPLE_N, len(os.listdir(path))))
    for fname in files:
        ratio = tissue_ratio(os.path.join(path, fname))
        tissue_records.append({'cluster': cl, 'size': '50px', 'tissue_ratio': ratio})

tissue_df = pd.DataFrame(tissue_records)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('2-A: Tissue Content Distribution per Cluster', fontweight='bold')

for ax, size_label in zip(axes, ['100px', '50px']):
    sub = tissue_df[tissue_df['size'] == size_label]
    for cl in IMAGE_DIRS_100.keys():
        vals = sub[sub['cluster'] == cl]['tissue_ratio']
        ax.hist(vals, bins=40, alpha=0.5, label=cl, density=True, color=COLORS[cl])
    ax.axvline(0.05, color='red', linestyle='--', label='threshold=0.05')
    ax.set_xlabel('Tissue Ratio')
    ax.set_ylabel('Density')
    ax.set_title(size_label, fontweight='bold')
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig(OUT_DIR + 'deep_2a_tissue_content.png', dpi=150, bbox_inches='tight')
plt.show()

# Print statistics of low-quality patches
print("\nLow quality patches (tissue_ratio < 0.05):")
for size_label in ['100px', '50px']:
    sub = tissue_df[tissue_df['size'] == size_label]
    n_low = (sub['tissue_ratio'] < 0.05).sum()
    print(f"  {size_label}: {n_low} / {len(sub)} ({100*n_low/len(sub):.1f}%)")



Low quality patches (tissue_ratio < 0.05):
  100px: 1 / 800 (0.1%)
  50px: 3 / 800 (0.4%)


The tissue ratios of all four types of cells were concentrated around 0.9-1.0, far exceeding the threshold of 0.05, indicating that most patches had abundant tissue content and the problem of blank patches was basically non-existent.

#### 2-B  50px vs 100px side-by-side (same cell)

In [16]:
pairs = []
for cl in IMAGE_DIRS_100.keys():
    shared = list(ids_both[cl])
    if shared:
        sampled = random.sample(shared, min(3, len(shared)))
        for cid in sampled:
            pairs.append((cl, cid))

fig, axes = plt.subplots(len(pairs), 2, figsize=(5, len(pairs) * 2.2))
fig.suptitle('2-B: Same Cell — 50px vs 100px', fontweight='bold')
if len(pairs) == 1:
    axes = [axes]

for i, (cl, cid) in enumerate(pairs):
    p50  = os.path.join(IMAGE_DIRS_50[cl],  f'cell_{cid}_50.png')
    p100 = os.path.join(IMAGE_DIRS_100[cl], f'cell_{cid}_100.png')
    axes[i][0].imshow(Image.open(p50));  axes[i][0].axis('off')
    axes[i][1].imshow(Image.open(p100)); axes[i][1].axis('off')
    axes[i][0].set_ylabel(f'{cl}\ncell {cid}', fontsize=7,
                           rotation=0, labelpad=80, va='center')
    if i == 0:
        axes[i][0].set_title('50px', fontweight='bold')
        axes[i][1].set_title('100px', fontweight='bold')

plt.tight_layout()
plt.savefig(OUT_DIR + 'deep_2b_50vs100.png', dpi=150, bbox_inches='tight')
plt.show()


#### 2-C  Random sample grid (8 per cluster, 100px)

In [17]:
fig, axes = plt.subplots(4, 8, figsize=(20, 10))
fig.suptitle('2-C: Random Sample — 8 patches per cluster (100px)', fontweight='bold')

for row, (cl, path) in enumerate(IMAGE_DIRS_100.items()):
    files = random.sample(os.listdir(path), 8)
    for col, fname in enumerate(files):
        img = Image.open(os.path.join(path, fname))
        axes[row, col].imshow(img)
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_title(cl.replace('_', '\n'), 
                                      fontsize=9, fontweight='bold',
                                      color=COLORS[cl], loc='left')

plt.tight_layout()
plt.savefig(OUT_DIR + 'deep_2c_sample_grid.png', dpi=150, bbox_inches='tight')
plt.show()

### More details for images 

1. shape → area, eccentricity, solidity (cell shape, size, compactness)
2. intensity → mean_intensity, std_intensity (color intensity)
3. texture → contrast, homogeneity (GLCM texture)

#### 2-D GRB Violin Plot

In [18]:
from skimage import color, feature, measure
import numpy as np

def extract_features(img_path):
    img = np.array(Image.open(img_path).convert('RGB'))
    gray = color.rgb2gray(img)
    
    # Strength Characteristic
    mean_intensity = gray.mean()
    std_intensity = gray.std()
    
    # Texture Feature (GLCM)
    from skimage.feature import graycomatrix, graycoprops
    gray_uint8 = (gray * 255).astype(np.uint8)
    glcm = graycomatrix(gray_uint8, [1], [0], 256, symmetric=True, normed=True)
    contrast = graycoprops(glcm, 'contrast')[0, 0]
    homogeneity = graycoprops(glcm, 'homogeneity')[0, 0]
    
    # Morphological Characteristics
    binary = gray < 0.7  # Simple Threshold Segmentation
    labeled = measure.label(binary)
    props = measure.regionprops(labeled)
    if props:
        largest = max(props, key=lambda r: r.area)
        area = largest.area
        eccentricity = largest.eccentricity
        solidity = largest.solidity
    
    else:
        area, eccentricity, solidity = np.nan, np.nan, np.nan
    
    return {
        'mean_intensity': mean_intensity,
        'std_intensity': std_intensity,
        'contrast': contrast,
        'homogeneity': homogeneity,
        'area': area,
        'eccentricity': eccentricity,
        'solidity': solidity,
    }


Mean Image:<br>
The average images of the four types of cells are almost exactly the same, all uniformly light purple, showing no structural or color differences. This indicates that there is no distinction in the overall color distribution among the four types of cells.<br>

RGB Violin:<br>
The distribution of the three channels is highly overlapping among the four types of cells, and the median values are almost the same.<br>

#### Cell-level Image Feature Extraction

In [19]:
SAMPLE_N = 300
random.seed(3888)

records_100 = []
for cl, path in IMAGE_DIRS_100.items():
    files = random.sample(os.listdir(path), min(SAMPLE_N, len(os.listdir(path))))
    for fname in files:
        cell_id = get_cell_id(fname)
        feats = extract_features(os.path.join(path, fname))
        feats['cluster'] = cl
        feats['cell_id'] = cell_id
        records_100.append(feats)
features_df_100 = pd.DataFrame(records_100)

records_50 = []
for cl, path in IMAGE_DIRS_50.items():
    files = random.sample(os.listdir(path), min(SAMPLE_N, len(os.listdir(path))))
    for fname in files:
        cell_id = get_cell_id(fname)
        feats = extract_features(os.path.join(path, fname))
        feats['cluster'] = cl
        feats['cell_id'] = cell_id
        records_50.append(feats)
features_df_50 = pd.DataFrame(records_50)

features_df = features_df_100

print("100px:", features_df_100.shape)
print("50px: ", features_df_50.shape)

100px: (1200, 9)
50px:  (1200, 9)


#### 2-E Quantification of feature separability - KS statistical heat map

In [20]:
from scipy.stats import ks_2samp
import seaborn as sns

feature_cols = ['mean_intensity', 'std_intensity', 'contrast', 'homogeneity', 'area', 'eccentricity', 'solidity']

clusters = ['DCIS_1', 'DCIS_2', 'Prolif_Invasive_Tumor', 'Invasive_Tumor']
pairs = [(clusters[i], clusters[j]) for i in range(len(clusters)) for j in range(i+1, len(clusters))]

ks_results = {}
for feat in feature_cols:
    ks_results[feat] = {}
    for c1, c2 in pairs:
        v1 = features_df[features_df['cluster'] == c1][feat].dropna()
        v2 = features_df[features_df['cluster'] == c2][feat].dropna()
        stat, _ = ks_2samp(v1, v2)
        ks_results[feat][f'{c1[:6]}\nvs\n{c2[:6]}'] = stat

ks_df = pd.DataFrame(ks_results).T  # features as rows, pairs as cols

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(ks_df, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax,
            vmin=0, vmax=1, linewidths=0.5)
ax.set_title('KS Statistic Heatmap\n(higher = more separable)', fontweight='bold')
ax.set_xlabel('Class Pair')
ax.set_ylabel('Feature')
plt.tight_layout()
plt.savefig(OUT_DIR + 'ks_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

main findings: <br>
1. The overall values are all very low (the highest being only 0.24), indicating that these four types of cells are generally difficult to distinguish based on these 7 characteristics.
2. The most crucial column - Prolif vs Invasi (the rightmost column):
The KS values for all features are the lowest in the entire table, with the highest being 0.11 (area) and the lowest being 0.04 (eccentricity). This indicates that prolif and invasive are almost unable to distinguish at the cellular level.

#### 2-F Feature space PCA + UMAP

In [21]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import umap

# Preparing Data
X = features_df[feature_cols].dropna()
y = features_df.loc[X.index, 'cluster']
X_scaled = StandardScaler().fit_transform(X)

# PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# UMAP
reducer = umap.UMAP(n_components=2, random_state=42)
X_umap = reducer.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Feature Space: PCA & UMAP', fontweight='bold')

for ax, coords, title in zip(axes, [X_pca, X_umap], ['PCA', 'UMAP']):
    for cl in clusters:
        mask = y == cl
        ax.scatter(coords[mask, 0], coords[mask, 1],
                   c=COLORS[cl], label=cl, alpha=0.5, s=10)
    ax.set_title(f'{title}\n(cell-level features)', fontweight='bold')
    ax.legend(fontsize=7, markerscale=2)
    if title == 'PCA':
        ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
        ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
    else:
        ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')

plt.tight_layout()
plt.savefig(OUT_DIR + 'feature_pca_umap.png', dpi=150, bbox_inches='tight')
plt.show()

2026-04-29 10:16:24.446694: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777457784.783579      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777457784.871413      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777457785.596022      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777457785.596082      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777457785.596085      16 computation_placer.cc:177] computation placer alr

1. PCA: The four types of cells are completely mixed together without any separation. PC1 explains 49.2% and PC2 explains 21.2%. Even when combining the two principal components, 70% of the variance cannot separate these four types of cells. This indicates that using these 7 cell-level features (intensity, texture, morphology) in a linear space cannot distinguish these four types of cells at all.
2. UMAP: Although UMAP can capture nonlinear structures, the four colors are still highly mixed and no obvious category clusters are observed.

#### 2-G Quantification of Feature Differences Between 50px and 100px

In [22]:
merged = features_df_100.merge(features_df_50, on=['cell_id', 'cluster'],
                                suffixes=('_100', '_50'))

# Calculate the difference for each feature
for feat in feature_cols:
    merged[f'delta_{feat}'] = merged[f'{feat}_100'] - merged[f'{feat}_50']

delta_cols = [f'delta_{f}' for f in feature_cols]

# Violin plot of deltas per cluster
fig, axes = plt.subplots(1, len(delta_cols), figsize=(20, 5))
fig.suptitle('Feature Change: 100px minus 50px (by cluster)', fontweight='bold')

for ax, col in zip(axes, delta_cols):
    data = [merged[merged['cluster'] == cl][col].dropna() for cl in clusters]
    parts = ax.violinplot(data, positions=range(len(clusters)), showmedians=True)
    for i, pc in enumerate(parts['bodies']):
        pc.set_facecolor(list(COLORS.values())[i])
        pc.set_alpha(0.7)
    ax.set_xticks(range(len(clusters)))
    ax.set_xticklabels([c[:6] for c in clusters], rotation=30, fontsize=7)
    ax.set_title(col.replace('delta_', ''), fontsize=8, fontweight='bold')
    ax.axhline(0, color='black', linestyle='--', linewidth=0.8)

plt.tight_layout()
plt.savefig(OUT_DIR + 'delta_features_50vs100.png', dpi=150, bbox_inches='tight')
plt.show()

# summary table
print("\nMean absolute delta per cluster:")
print(merged.groupby('cluster')[delta_cols].apply(lambda x: x.abs().mean()).round(3))


Mean absolute delta per cluster:
                       delta_mean_intensity  delta_std_intensity  \
cluster                                                            
DCIS_1                                0.019                0.009   
DCIS_2                                0.014                0.007   
Invasive_Tumor                        0.029                0.014   
Prolif_Invasive_Tumor                 0.019                0.009   

                       delta_contrast  delta_homogeneity  delta_area  \
cluster                                                                
DCIS_1                          4.792              0.013   21842.200   
DCIS_2                          2.312              0.016   16372.750   
Invasive_Tumor                  2.584              0.007   21808.000   
Prolif_Invasive_Tumor           3.942              0.012   18638.353   

                       delta_eccentricity  delta_solidity  
cluster                                                    
DCIS

This graph shows the differences (100px minus 50px) of each feature for the same cell. The larger the difference, the greater the impact of the patch size on that feature.<br>
main finding:<br>
1. The area is the most obvious - the differences in all categories are large and positive, indicating that the cell morphology area captured by the 100px patch is much larger than that of the 50px patch. This is in line with expectations; the larger the patch, the more tissue it contains.<br>
2. The Invasive_Tumor (in blue) has a significantly narrower distribution, indicating that the change in patch size from 50px to 100px has little and stable impact on its characteristics. The other three types (DCIS_1, DCIS_2, Prolif) have a wider distribution, suggesting that the characteristics of these three types of cells are more affected by the patch size and less stable.<br>
3. The "Invasive_Tumor" is not sensitive to changes in the context. This might be because it is the most numerous (with over 34,000 instances), and in both 50px and 100px, the surrounding environment is the same, resulting in small changes in features. On the other hand, the "Prolif" distribution is wide, indicating that it responds to patch size.

#### HOG

In [23]:
from skimage.feature import hog
from skimage import color
import numpy as np
import pandas as pd
from PIL import Image
import os, random
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import umap

random.seed(42)
SAMPLE_N = 300

def extract_hog(img_path, pixels_per_cell=(8, 8), cells_per_block=(2, 2), orientations=9):
    img = np.array(Image.open(img_path).convert('RGB').resize((64, 64)))
    gray = color.rgb2gray(img)
    features = hog(gray, orientations=orientations,
                   pixels_per_cell=pixels_per_cell,
                   cells_per_block=cells_per_block,
                   block_norm='L2-Hys')
    return features

# Extract HOG features
hog_records = []
for cl, path in IMAGE_DIRS_100.items():
    files = random.sample(os.listdir(path), min(SAMPLE_N, len(os.listdir(path))))
    for fname in files:
        feats = extract_hog(os.path.join(path, fname))
        hog_records.append({'cluster': cl, 'hog': feats})

hog_df = pd.DataFrame(hog_records)
X_hog = np.stack(hog_df['hog'].values)
y_hog = hog_df['cluster'].values

print(f"HOG feature vector length: {X_hog.shape[1]}")

# PCA
X_scaled = StandardScaler().fit_transform(X_hog)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# UMAP
reducer = umap.UMAP(n_components=2, random_state=42)
X_umap = reducer.fit_transform(X_scaled)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('HOG Feature Space: PCA & UMAP', fontweight='bold')

for ax, coords, title in zip(axes, [X_pca, X_umap], ['PCA', 'UMAP']):
    for cl in IMAGE_DIRS_100.keys():
        mask = y_hog == cl
        ax.scatter(coords[mask, 0], coords[mask, 1],
                   c=COLORS[cl], label=cl, alpha=0.5, s=10)
    ax.set_title(f'{title}\n(HOG features)', fontweight='bold')
    ax.legend(fontsize=7, markerscale=2)
    if title == 'PCA':
        ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
        ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
    else:
        ax.set_xlabel('UMAP 1')
        ax.set_ylabel('UMAP 2')

plt.tight_layout()
plt.savefig(OUT_DIR + 'hog_pca_umap.png', dpi=150, bbox_inches='tight')
plt.show()

# visualision
sample_path = os.path.join(IMAGE_DIRS_100['Prolif_Invasive_Tumor'],
                            os.listdir(IMAGE_DIRS_100['Prolif_Invasive_Tumor'])[0])
img_sample = np.array(Image.open(sample_path).convert('RGB').resize((64, 64)))
gray_sample = color.rgb2gray(img_sample)
_, hog_img = hog(gray_sample, orientations=9, pixels_per_cell=(8, 8),
                  cells_per_block=(2, 2), block_norm='L2-Hys', visualize=True)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(img_sample)
axes[0].set_title('Original (Prolif)', fontweight='bold')
axes[0].axis('off')
axes[1].imshow(hog_img, cmap='gray')
axes[1].set_title('HOG visualization', fontweight='bold')
axes[1].axis('off')
plt.tight_layout()
plt.savefig(OUT_DIR + 'hog_example.png', dpi=150, bbox_inches='tight')
plt.show()

HOG feature vector length: 1764


HOG example image:<br>
HOG has successfully captured the edges and gradient directions of the cell nucleus. The dark nucleus outline can be seen with a bright direction gradient in the HOG image, indicating that HOG is responsive to the structural information of H&E images.<br>

PCA & UMAP:<br>
The results of PCA are poor. PC1 explains only 3.4%, and PC2 only 2.5%, adding up to less than 6%. This indicates that the HOG feature dimension is very high but the variance is extremely dispersed. Linear projection basically loses all the information, and PCA is not suitable for visualizing HOG. UMAP is relatively more reasonable, but the four types of cells are still highly mixed, with no obvious clustering separation, and the four colors are completely overlapping.<br>

In [24]:
fig, ax = plt.subplots(figsize=(12, 9))

unique_clusters = xenium['Cluster'].unique()
cmap = plt.cm.get_cmap('tab20', len(unique_clusters))
cluster_color_map = {cl: cmap(i) for i, cl in enumerate(unique_clusters)}

for cl in unique_clusters:
    sub = xenium[xenium['Cluster'] == cl]
    ax.scatter(sub['x_centroid'], sub['y_centroid'],
               c=[cluster_color_map[cl]], s=0.3, alpha=0.6,
               label=cl, rasterized=True)

ax.set_xlabel('X centroid (µm)')
ax.set_ylabel('Y centroid (µm)')
ax.set_title('Spatial Map — All 20 Cell Types', fontweight='bold')
ax.invert_yaxis()
ax.legend(markerscale=8, fontsize=7, loc='upper right',
          ncol=2, bbox_to_anchor=(1.35, 1))
plt.tight_layout()
plt.savefig(OUT_DIR + 'spatial_all20.png', dpi=150, bbox_inches='tight')
plt.show()